In [1]:
# ================================================================
# CELL 1 — Configuration
# ================================================================

# -----------------------------
# Hugging Face Model Names
# -----------------------------

# Base embedding model
BASE_MODEL = "Qwen/Qwen3-Embedding-0.6B"

# Fine-tuned embedding model
FT_MODEL = "boq-retrieval-team/boq-retrieval-team"

# -----------------------------
# Hardware
# -----------------------------

DEVICE = "cuda"          # "cuda" or "cpu"
USE_FP16 = True          # Faster inference on supported GPUs

# -----------------------------
# Embedding Configuration
# -----------------------------

BATCH_SIZE = 32          # Increase if GPU memory allows
NORMALIZE = True         # L2 Normalize embeddings

# -----------------------------
# Retrieval Configuration
# -----------------------------

TOP_K = [1, 3, 5, 10]

# -----------------------------
# Evaluation Metrics
# -----------------------------

# Ranking Metrics
EVALUATE_RECALL = True
EVALUATE_PRECISION = True
EVALUATE_MRR = True
EVALUATE_MAP = True

# Error Metrics
EVALUATE_ERROR_RATE = True

# Similarity Metrics
EVALUATE_COSINE = True
EVALUATE_DOT_PRODUCT = True
EVALUATE_EUCLIDEAN = True

# Query Analysis
EVALUATE_QUERY_DIFFICULTY = True

# Efficiency Metrics
EVALUATE_RETRIEVAL_TIME = True
EVALUATE_ENCODING_TIME = True
EVALUATE_LATENCY = True
EVALUATE_THROUGHPUT = True
EVALUATE_MEMORY = True

# -----------------------------
# Random Seed
# -----------------------------

SEED = 42

# -----------------------------
# Output Directories
# -----------------------------

OUTPUT_DIR = "benchmark_output"
CACHE_DIR = "embedding_cache"

# -----------------------------
# Visualization
# -----------------------------

FIG_DPI = 200
FIG_SIZE = (8, 5)
SAVE_FIGURES = True

# -----------------------------
# Report Configuration
# -----------------------------

EXPORT_CSV = True
EXPORT_EXCEL = True
EXPORT_PDF = True

# -----------------------------
# Display Configuration
# -----------------------------

SHOW_TABLES = True
SHOW_GRAPHS = True

# ================================================================
# Configuration Summary
# ================================================================

print("=" * 70)
print("Benchmark Evaluation Configuration")
print("=" * 70)

print(f"Base Model           : {BASE_MODEL}")
print(f"Fine-tuned Model     : {FT_MODEL}")
print(f"Device               : {DEVICE}")
print(f"Batch Size           : {BATCH_SIZE}")
print(f"Normalize Embeddings : {NORMALIZE}")
print(f"Top-K Values         : {TOP_K}")

print("\nEvaluation Metrics")
print("-" * 70)
print("Ranking Metrics      : Recall | Precision | MRR | MAP")
print("Error Metrics        : Error Rate")
print("Similarity Metrics   : Cosine | Dot Product | Euclidean")
print("Query Analysis       : Query Difficulty")
print("Efficiency Metrics   : Encoding Time | Retrieval Time | Latency | Throughput | Memory")

print("\nOutput")
print("-" * 70)
print(f"Results Folder       : {OUTPUT_DIR}")
print(f"Embedding Cache      : {CACHE_DIR}")

print("=" * 70)

Benchmark Evaluation Configuration
Base Model           : Qwen/Qwen3-Embedding-0.6B
Fine-tuned Model     : boq-retrieval-team/boq-retrieval-team
Device               : cuda
Batch Size           : 32
Normalize Embeddings : True
Top-K Values         : [1, 3, 5, 10]

Evaluation Metrics
----------------------------------------------------------------------
Ranking Metrics      : Recall | Precision | MRR | MAP
Error Metrics        : Error Rate
Similarity Metrics   : Cosine | Dot Product | Euclidean
Query Analysis       : Query Difficulty
Efficiency Metrics   : Encoding Time | Retrieval Time | Latency | Throughput | Memory

Output
----------------------------------------------------------------------
Results Folder       : benchmark_output
Embedding Cache      : embedding_cache


In [6]:
# ================================================================
# CELL 2 — Install Dependencies (VS Code)
# Run ONLY once
# ================================================================

import sys
import subprocess

packages = [
    "torch",
    "torchvision",
    "torchaudio",
    "transformers>=4.53.0",
    "sentence-transformers>=5.0.0",
    "accelerate>=1.8.0",
    "huggingface_hub>=0.34.0",
    "datasets>=3.0.0",
    "pdfplumber",
    "PyMuPDF",
    "faiss-cpu",
    "scikit-learn",
    "matplotlib",
    "plotly",
    "umap-learn",
    "openpyxl",
    "reportlab",
    "tqdm",
    "groq",
    "pandas",
    "numpy"
]

print("=" * 70)
print("Installing Required Packages...")
print("=" * 70)

for package in packages:
    print(f"Installing {package}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", package])

print("\n" + "=" * 70)
print("✅ All dependencies installed successfully.")
print("⚠️ Restart the VS Code kernel before running the next cell.")
print("=" * 70)

Installing Required Packages...
Installing torch...
Installing torchvision...
Installing torchaudio...
Installing transformers>=4.53.0...
Installing sentence-transformers>=5.0.0...
Installing accelerate>=1.8.0...
Installing huggingface_hub>=0.34.0...
Installing datasets>=3.0.0...
Installing pdfplumber...
Installing PyMuPDF...
Installing faiss-cpu...
Installing scikit-learn...
Installing matplotlib...
Installing plotly...
Installing umap-learn...
Installing openpyxl...
Installing reportlab...
Installing tqdm...
Installing groq...
Installing pandas...
Installing numpy...

✅ All dependencies installed successfully.
⚠️ Restart the VS Code kernel before running the next cell.


In [ ]:
import torch
import transformers
import sentence_transformers

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

print("Transformers:", transformers.__version__)
print("SentenceTransformers:", sentence_transformers.__version__)

In [ ]:
# ================================================================
# CELL 3 — Imports & Environment Verification
# ================================================================

import os
import gc
import json
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import faiss
import fitz  # PyMuPDF
import pdfplumber

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from tqdm.auto import tqdm
from datasets import Dataset
from sentence_transformers import SentenceTransformer

# Removed:
# from sklearn.metrics import ndcg_score

from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize
import umap.umap_ as umap

warnings.filterwarnings("ignore")

# ------------------------------------------------
# Random Seed
# ------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------
# Device Detection
# ------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 60)
print("Environment Verification")
print("=" * 60)

print(f"Python               : {os.sys.version.split()[0]}")
print(f"Torch                : {torch.__version__}")
print(f"CUDA Available       : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU                  : {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version         : {torch.version.cuda}")
else:
    print("GPU                  : Not Available (Running on CPU)")

print(f"SentenceTransformers : {SentenceTransformer.__module__.split('.')[0]}")
print(f"FAISS                : {faiss.__version__}")
print(f"PyMuPDF              : {fitz.version[0]}")
print(f"pdfplumber           : {pdfplumber.__version__}")

# ------------------------------------------------
# Create Required Directories
# ------------------------------------------------

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

Path(f"{OUTPUT_DIR}/plots").mkdir(parents=True, exist_ok=True)
Path(f"{OUTPUT_DIR}/reports").mkdir(parents=True, exist_ok=True)
Path(f"{OUTPUT_DIR}/tables").mkdir(parents=True, exist_ok=True)
Path(f"{CACHE_DIR}/embeddings").mkdir(parents=True, exist_ok=True)
Path(f"{CACHE_DIR}/faiss").mkdir(parents=True, exist_ok=True)

print("\nDirectories Created:")
print(f"  📁 {OUTPUT_DIR}")
print(f"  📁 {CACHE_DIR}")

# ------------------------------------------------
# GPU Optimization
# ------------------------------------------------

if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

    print("\nGPU optimizations enabled.")

print("=" * 60)
print("✅ Cell 3 completed successfully.")
print("=" * 60)

In [ ]:
from huggingface_hub import login, HfApi

# ------------------------------------------------
# Login to Hugging Face
# ------------------------------------------------

try:
    login(skip_if_logged_in=True)
except Exception as e:
    print(f"Login skipped: {e}")

# ------------------------------------------------
# Repository Validation
# ------------------------------------------------

BASE_MODEL = globals().get("BASE_MODEL", "BAAI/bge-small-en-v1.5")
FT_MODEL = globals().get("FT_MODEL", "boq-retrieval-team/boq-retrieval-team")

api = HfApi()

print("=" * 60)
print("Validating Hugging Face Repositories")
print("=" * 60)

# Base model
try:
    api.model_info(BASE_MODEL)
    print(f"✅ Base Model Found      : {BASE_MODEL}")
except Exception as e:
    raise Exception(f"Base model not accessible:\n{e}")

# Fine-tuned model
try:
    api.model_info(FT_MODEL)
    print(f"✅ Fine-Tuned Model Found: {FT_MODEL}")
except Exception as e:
    raise Exception(
        f"Fine-tuned model not accessible.\n"
        f"Check repository name or permissions.\n\n{e}"
    )

print("=" * 60)
print("✅ Hugging Face setup completed successfully.")
print("=" * 60)

In [ ]:
# ================================================================
# CELL 5 — Upload BOQ PDFs
# ================================================================

from google.colab import files
from pathlib import Path
import shutil
import os

# ------------------------------------------------
# Create PDF Directory
# ------------------------------------------------

PDF_DIR = Path("boq_pdfs")
PDF_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("Upload BOQ PDF Files")
print("=" * 60)
print("Select one or more BOQ PDFs.")
print()

# ------------------------------------------------
# Upload Files
# ------------------------------------------------

uploaded = files.upload()

if len(uploaded) == 0:
    raise Exception("No PDF files uploaded.")

# ------------------------------------------------
# Save Files
# ------------------------------------------------

pdf_files = []

for filename in uploaded.keys():

    if not filename.lower().endswith(".pdf"):
        print(f"Skipping non-PDF file: {filename}")
        continue

    destination = PDF_DIR / filename

    shutil.move(filename, destination)

    pdf_files.append(destination)

# ------------------------------------------------
# Summary
# ------------------------------------------------

print("\n" + "=" * 60)
print(f"Successfully uploaded {len(pdf_files)} PDF(s)")
print("=" * 60)

for i, pdf in enumerate(pdf_files, start=1):
    size_mb = os.path.getsize(pdf) / (1024 * 1024)
    print(f"{i}. {pdf.name} ({size_mb:.2f} MB)")

print("=" * 60)

In [ ]:
# ================================================================
# CELL 6 — Extract BOQ Tables
# ================================================================

import pdfplumber
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

print("=" * 60)
print("Extracting BOQ Tables")
print("=" * 60)

all_rows = []

# ------------------------------------------------
# Helper Function
# ------------------------------------------------

def clean_cell(cell):
    if cell is None:
        return ""
    return " ".join(str(cell).replace("\n", " ").split()).strip()

# ------------------------------------------------
# Process Every PDF
# ------------------------------------------------

for pdf_path in tqdm(pdf_files):

    with pdfplumber.open(pdf_path) as pdf:

        for page_number, page in enumerate(pdf.pages, start=1):

            try:
                tables = page.extract_tables()

            except Exception:
                tables = []

            # ----------------------------------------
            # Process Tables
            # ----------------------------------------

            if tables:

                for table_id, table in enumerate(tables):

                    if table is None or len(table) == 0:
                        continue

                    for row_id, row in enumerate(table):

                        if row is None:
                            continue

                        cleaned = [clean_cell(c) for c in row]

                        if all(c == "" for c in cleaned):
                            continue

                        all_rows.append({

                            "pdf_name": pdf_path.name,

                            "page": page_number,

                            "table": table_id + 1,

                            "row": row_id + 1,

                            "cells": cleaned

                        })

            # ----------------------------------------
            # Fallback (No Tables)
            # ----------------------------------------

            else:

                text = page.extract_text()

                if text:

                    for line_id, line in enumerate(text.split("\n")):

                        line = clean_cell(line)

                        if len(line) < 5:
                            continue

                        all_rows.append({

                            "pdf_name": pdf_path.name,

                            "page": page_number,

                            "table": 0,

                            "row": line_id + 1,

                            "cells": [line]

                        })

# ------------------------------------------------
# Create DataFrame
# ------------------------------------------------

raw_df = pd.DataFrame(all_rows)

print("\nExtraction Complete")
print(f"Documents Extracted : {len(raw_df):,}")
print(f"PDFs Processed      : {len(pdf_files)}")

display(raw_df.head(10))

In [ ]:
# ================================================================
# CELL 7 — Clean Tables & Build Retrieval Corpus
# ================================================================

import re
import pandas as pd

print("=" * 60)
print("Building Retrieval Corpus")
print("=" * 60)

HEADER_KEYWORDS = {
    "item", "description", "desc", "unit", "qty",
    "quantity", "rate", "amount", "total", "sr",
    "s.no", "no", "remarks"
}


def normalize(text):
    return re.sub(r"\s+", " ", str(text)).strip()


def is_header_row(cells):
    joined = " ".join(c.lower() for c in cells)
    hits = sum(k in joined for k in HEADER_KEYWORDS)
    return hits >= 2


documents = []

for idx, row in raw_df.iterrows():

    cells = [normalize(c) for c in row["cells"]]

    # Remove empty cells
    cells = [c for c in cells if c != ""]

    if len(cells) == 0:
        continue

    # Skip table headers
    if is_header_row(cells):
        continue

    # ------------------------------------------------
    # Description
    # ------------------------------------------------

    if len(cells) >= 2:
        description = cells[1]
    else:
        description = cells[0]

    description = normalize(description)

    if len(description) < 5:
        continue

    # ------------------------------------------------
    # Metadata
    # ------------------------------------------------

    unit = cells[2] if len(cells) > 2 else ""
    qty = cells[3] if len(cells) > 3 else ""
    rate = cells[4] if len(cells) > 4 else ""
    amount = cells[5] if len(cells) > 5 else ""

    documents.append({

        "doc_id": f"DOC_{len(documents):06d}",

        "text": description,

        "pdf_name": row["pdf_name"],

        "page": row["page"],

        "table": row["table"],

        "row": row["row"],

        "unit": unit,

        "qty": qty,

        "rate": rate,

        "amount": amount

    })

# ------------------------------------------------
# DataFrame
# ------------------------------------------------

corpus_df = pd.DataFrame(documents)

print(f"Total Retrieval Documents : {len(corpus_df):,}")

display(corpus_df.head(10))

# ------------------------------------------------
# Save
# ------------------------------------------------

corpus_path = Path(OUTPUT_DIR) / "tables" / "retrieval_corpus.csv"

corpus_df.to_csv(corpus_path, index=False)

print()
print(f"Saved corpus to:\n{corpus_path}")

print("=" * 60)
print("✅ Retrieval corpus ready.")
print("=" * 60)

In [ ]:
# ================================================================
# CELL 8 — Generate Evaluation Queries (One Query Per Document)
# ================================================================

import random
import pandas as pd

print("="*70)
print("Generating Evaluation Queries")
print("="*70)

random.seed(SEED)

QUERY_TEMPLATES = [
    "Find BOQ item for {}",
    "Locate {}",
    "Search for {}",
    "Show me {}",
    "Where is {}",
    "Retrieve {}",
    "BOQ entry for {}"
]

queries = []
ground_truth = []

rows = []

for idx, row in corpus_df.reset_index(drop=True).iterrows():

    desc = str(row["text"]).strip()

    # Never skip documents
    if desc == "":
        desc = f"Document {idx}"

    q = random.choice(QUERY_TEMPLATES).format(desc)

    queries.append(q)
    ground_truth.append(idx)

    rows.append({
        "query_id": f"Q{idx:06d}",
        "query": q,
        "ground_truth": idx,
        "description": desc,
        "pdf_name": row["pdf_name"],
        "page": row["page"]
    })

queries_df = pd.DataFrame(rows)

print(f"Corpus        : {len(corpus_df)}")
print(f"Queries       : {len(queries)}")
print(f"Ground Truth  : {len(ground_truth)}")

display(queries_df.head())

queries_df.to_csv(
    Path(OUTPUT_DIR)/"tables"/"evaluation_queries.csv",
    index=False
)

print("\n✓ Evaluation dataset created.")

In [ ]:
# ================================================================
# CELL 9 — Load Embedding Models
# ================================================================

from sentence_transformers import SentenceTransformer
import torch
import time

print("=" * 60)
print("Loading Embedding Models")
print("=" * 60)

# ------------------------------------------------
# Configuration
# ------------------------------------------------

BASE_MODEL = globals().get("BASE_MODEL", "Qwen/Qwen3-Embedding-0.6B")
FT_MODEL = globals().get("FT_MODEL", "boq-retrieval-team/boq-retrieval-team")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NORMALIZE = globals().get("NORMALIZE", True)

# ------------------------------------------------
# Load Base Model
# ------------------------------------------------

t0 = time.time()

base_model = SentenceTransformer(
    BASE_MODEL,
    device=DEVICE
)

BASE_MODEL_LOAD_TIME = time.time() - t0

print(f"✓ Base Model Loaded ({BASE_MODEL_LOAD_TIME:.2f}s)")

# ------------------------------------------------
# Load Fine-Tuned Model
# ------------------------------------------------

t0 = time.time()

ft_model = SentenceTransformer(
    FT_MODEL,
    device=DEVICE
)

FT_MODEL_LOAD_TIME = time.time() - t0

print(f"✓ Fine-Tuned Model Loaded ({FT_MODEL_LOAD_TIME:.2f}s)")

# ------------------------------------------------
# Verify Embedding Dimension
# ------------------------------------------------

sample = ["Concrete grade M25"]

base_vec = base_model.encode(
    sample,
    convert_to_numpy=True,
    normalize_embeddings=NORMALIZE
)

ft_vec = ft_model.encode(
    sample,
    convert_to_numpy=True,
    normalize_embeddings=NORMALIZE
)

EMBED_DIM = base_vec.shape[1]

assert base_vec.shape == ft_vec.shape, \
    "Embedding dimensions do not match!"

print()
print(f"Embedding Dimension : {EMBED_DIM}")
print(f"Device              : {DEVICE}")

if DEVICE == "cuda":
    print(f"GPU                 : {torch.cuda.get_device_name(0)}")

print("\nModel Loading Summary")
print("-" * 60)
print(f"Base Model Load Time      : {BASE_MODEL_LOAD_TIME:.2f} sec")
print(f"Fine-Tuned Model Load Time: {FT_MODEL_LOAD_TIME:.2f} sec")

print("=" * 60)
print("✅ Both models loaded successfully.")
print("=" * 60)

In [ ]:
# ================================================================
# CELL 10 — Smart Embedding Cache + Efficiency Metrics
# ================================================================

import gc
import time
import numpy as np
from pathlib import Path
import torch

print("=" * 70)
print("Embedding Corpus")
print("=" * 70)

CACHE_PATH = Path(CACHE_DIR)
CACHE_PATH.mkdir(parents=True, exist_ok=True)

BASE_CACHE = CACHE_PATH / "base_embeddings.npy"
FT_CACHE = CACHE_PATH / "ft_embeddings.npy"

documents = corpus_df["text"].astype(str).tolist()
EXPECTED_DOCS = len(documents)

# --------------------------------------------------------
# Encoder
# --------------------------------------------------------

def encode_model(model, texts, batch_size):

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    gc.collect()

    start = time.time()

    with torch.inference_mode():

        embeddings = model.encode(
            texts,
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=NORMALIZE,
            show_progress_bar=True
        )

    elapsed = time.time() - start

    embeddings = embeddings.astype(np.float32)

    throughput = len(texts) / elapsed if elapsed > 0 else 0

    memory_mb = embeddings.nbytes / (1024 ** 2)

    avg_norm = np.mean(np.linalg.norm(embeddings, axis=1))

    return embeddings, elapsed, throughput, memory_mb, avg_norm


# ================================================================
# BASE MODEL
# ================================================================

rebuild = False

if BASE_CACHE.exists():

    base_embeddings = np.load(BASE_CACHE)

    if base_embeddings.shape[0] == EXPECTED_DOCS:

        print("✓ Loaded cached BASE embeddings")

        BASE_ENCODING_TIME = 0.0
        BASE_THROUGHPUT = 0.0

    else:

        print("⚠ BASE cache mismatch")
        rebuild = True

else:

    rebuild = True

if rebuild:

    print("\nEncoding BASE model...")

    (
        base_embeddings,
        BASE_ENCODING_TIME,
        BASE_THROUGHPUT,
        BASE_MEMORY_MB,
        BASE_AVG_NORM,
    ) = encode_model(
        base_model,
        documents,
        BATCH_SIZE,
    )

    np.save(BASE_CACHE, base_embeddings)

    print(f"✓ BASE encoded in {BASE_ENCODING_TIME:.2f} sec")

else:

    BASE_MEMORY_MB = base_embeddings.nbytes / (1024 ** 2)
    BASE_AVG_NORM = np.mean(np.linalg.norm(base_embeddings, axis=1))

if torch.cuda.is_available():
    torch.cuda.empty_cache()

gc.collect()

# ================================================================
# FINE-TUNED MODEL
# ================================================================

rebuild = False

if FT_CACHE.exists():

    ft_embeddings = np.load(FT_CACHE)

    if ft_embeddings.shape[0] == EXPECTED_DOCS:

        print("✓ Loaded cached FT embeddings")

        FT_ENCODING_TIME = 0.0
        FT_THROUGHPUT = 0.0

    else:

        print("⚠ FT cache mismatch")
        rebuild = True

else:

    rebuild = True

if rebuild:

    print("\nEncoding Fine-Tuned model...")

    (
        ft_embeddings,
        FT_ENCODING_TIME,
        FT_THROUGHPUT,
        FT_MEMORY_MB,
        FT_AVG_NORM,
    ) = encode_model(
        ft_model,
        documents,
        BATCH_SIZE,
    )

    np.save(FT_CACHE, ft_embeddings)

    print(f"✓ FT encoded in {FT_ENCODING_TIME:.2f} sec")

else:

    FT_MEMORY_MB = ft_embeddings.nbytes / (1024 ** 2)
    FT_AVG_NORM = np.mean(np.linalg.norm(ft_embeddings, axis=1))

assert base_embeddings.shape == ft_embeddings.shape

# ================================================================
# Efficiency Metrics Summary
# ================================================================

efficiency_df = np.round(
    np.array([
        [
            BASE_ENCODING_TIME,
            BASE_THROUGHPUT,
            BASE_MEMORY_MB,
            BASE_AVG_NORM,
        ],
        [
            FT_ENCODING_TIME,
            FT_THROUGHPUT,
            FT_MEMORY_MB,
            FT_AVG_NORM,
        ],
    ]),
    4,
)

import pandas as pd

efficiency_df = pd.DataFrame(
    efficiency_df,
    index=["Base Model", "Fine-Tuned Model"],
    columns=[
        "Encoding Time (s)",
        "Throughput (docs/s)",
        "Embedding Memory (MB)",
        "Average Embedding Norm",
    ],
)

print("\n" + "=" * 70)
print("Embedding Summary")
print("=" * 70)

print(f"Documents           : {EXPECTED_DOCS:,}")
print(f"Embedding Dimension : {base_embeddings.shape[1]}")
print(f"Base Shape          : {base_embeddings.shape}")
print(f"FT Shape            : {ft_embeddings.shape}")

print("\nEfficiency Metrics")
display(efficiency_df)

efficiency_df.to_csv(
    Path(OUTPUT_DIR) / "tables" / "embedding_efficiency_metrics.csv"
)

print("=" * 70)
print("✓ Embeddings Ready")

In [ ]:
# ================================================================
# CELL 11 — Build FAISS Indices + Efficiency Metrics
# ================================================================

import faiss
import time
import os
from pathlib import Path

print("=" * 70)
print("Building FAISS Index")
print("=" * 70)

CACHE_PATH = Path(CACHE_DIR)
CACHE_PATH.mkdir(parents=True, exist_ok=True)

BASE_INDEX_FILE = CACHE_PATH / "base_faiss.index"
FT_INDEX_FILE   = CACHE_PATH / "ft_faiss.index"

EXPECTED_SIZE = len(corpus_df)

# --------------------------------------------------------
# Helper
# --------------------------------------------------------

def build_faiss_index(embeddings, save_path):

    start = time.time()

    dim = embeddings.shape[1]

    index = faiss.IndexFlatIP(dim)

    index.add(embeddings.astype(np.float32))

    faiss.write_index(index, str(save_path))

    elapsed = time.time() - start

    file_size_mb = os.path.getsize(save_path) / (1024 ** 2)

    return index, elapsed, file_size_mb

# ========================================================
# BASE INDEX
# ========================================================

rebuild_base = True

if BASE_INDEX_FILE.exists():

    print("Found cached Base index.")

    temp = faiss.read_index(str(BASE_INDEX_FILE))

    if temp.ntotal == EXPECTED_SIZE:

        print("✓ Cache matches corpus.")

        base_index = temp
        rebuild_base = False

        BASE_INDEX_BUILD_TIME = 0.0
        BASE_INDEX_SIZE_MB = os.path.getsize(BASE_INDEX_FILE) / (1024 ** 2)

    else:

        print("⚠ Cache mismatch.")
        print(f" Cached : {temp.ntotal:,}")
        print(f" Current: {EXPECTED_SIZE:,}")
        print(" Rebuilding Base index...")

if rebuild_base:

    (
        base_index,
        BASE_INDEX_BUILD_TIME,
        BASE_INDEX_SIZE_MB
    ) = build_faiss_index(
        base_embeddings,
        BASE_INDEX_FILE
    )

    print(f"✓ Base index built in {BASE_INDEX_BUILD_TIME:.2f} sec")

# ========================================================
# FINE-TUNED INDEX
# ========================================================

rebuild_ft = True

if FT_INDEX_FILE.exists():

    print("\nFound cached Fine-Tuned index.")

    temp = faiss.read_index(str(FT_INDEX_FILE))

    if temp.ntotal == EXPECTED_SIZE:

        print("✓ Cache matches corpus.")

        ft_index = temp
        rebuild_ft = False

        FT_INDEX_BUILD_TIME = 0.0
        FT_INDEX_SIZE_MB = os.path.getsize(FT_INDEX_FILE) / (1024 ** 2)

    else:

        print("⚠ Cache mismatch.")
        print(f" Cached : {temp.ntotal:,}")
        print(f" Current: {EXPECTED_SIZE:,}")
        print(" Rebuilding Fine-Tuned index...")

if rebuild_ft:

    (
        ft_index,
        FT_INDEX_BUILD_TIME,
        FT_INDEX_SIZE_MB
    ) = build_faiss_index(
        ft_embeddings,
        FT_INDEX_FILE
    )

    print(f"✓ Fine-Tuned index built in {FT_INDEX_BUILD_TIME:.2f} sec")

# ========================================================
# Verification
# ========================================================

assert base_index.ntotal == EXPECTED_SIZE
assert ft_index.ntotal == EXPECTED_SIZE

print("\n" + "=" * 70)
print("FAISS Summary")
print("=" * 70)

print(f"Corpus Documents : {EXPECTED_SIZE:,}")
print(f"Base Index Size  : {base_index.ntotal:,}")
print(f"FT Index Size    : {ft_index.ntotal:,}")

# ========================================================
# Efficiency Metrics
# ========================================================

faiss_metrics_df = pd.DataFrame({
    "Model": ["Base Model", "Fine-Tuned Model"],
    "Index Build Time (s)": [
        BASE_INDEX_BUILD_TIME,
        FT_INDEX_BUILD_TIME
    ],
    "Index File Size (MB)": [
        BASE_INDEX_SIZE_MB,
        FT_INDEX_SIZE_MB
    ],
    "Indexed Documents": [
        base_index.ntotal,
        ft_index.ntotal
    ]
})

print("\nFAISS Efficiency Metrics")
display(faiss_metrics_df)

faiss_metrics_df.to_csv(
    Path(OUTPUT_DIR) / "tables" / "faiss_efficiency_metrics.csv",
    index=False
)

print("=" * 70)
print("✓ FAISS indices are ready.")
print("=" * 70)

In [ ]:
# ================================================================
# CELL 12 — Semantic Search Engine (Enhanced)
# ================================================================

import time
import torch
import pandas as pd

print("=" * 70)
print("Building Retrieval Engine")
print("=" * 70)

# --------------------------------------------------------
# Generic Search Function
# --------------------------------------------------------

def semantic_search(
    query,
    model,
    index,
    corpus_df,
    top_k=10
):
    """
    Semantic Search using FAISS.

    Returns
    -------
    results_df : DataFrame
        Retrieved documents.

    metrics : dict
        Retrieval efficiency metrics.
    """

    # ---------------------------------------------
    # Encode Query
    # ---------------------------------------------

    encode_start = time.perf_counter()

    with torch.inference_mode():

        query_embedding = model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=NORMALIZE,
            show_progress_bar=False
        ).astype("float32")

    encode_time = time.perf_counter() - encode_start

    # ---------------------------------------------
    # FAISS Search
    # ---------------------------------------------

    search_start = time.perf_counter()

    scores, ids = index.search(query_embedding, top_k)

    retrieval_time = time.perf_counter() - search_start

    total_time = encode_time + retrieval_time

    throughput = (
        1 / total_time
        if total_time > 0
        else 0
    )

    # ---------------------------------------------
    # Build Results
    # ---------------------------------------------

    results = corpus_df.iloc[ids[0]].copy()

    results.insert(
        0,
        "Rank",
        range(1, len(results) + 1)
    )

    results["Similarity Score"] = scores[0]

    metrics = {

        "encoding_time": encode_time,

        "retrieval_time": retrieval_time,

        "total_time": total_time,

        "throughput": throughput,

        "top_similarity": float(scores[0][0]),

        "average_similarity": float(scores[0].mean()),

        "minimum_similarity": float(scores[0].min())

    }

    return results.reset_index(drop=True), metrics


# --------------------------------------------------------
# Wrapper Functions
# --------------------------------------------------------

def search_base(query, top_k=10):

    return semantic_search(
        query=query,
        model=base_model,
        index=base_index,
        corpus_df=corpus_df,
        top_k=top_k
    )


def search_ft(query, top_k=10):

    return semantic_search(
        query=query,
        model=ft_model,
        index=ft_index,
        corpus_df=corpus_df,
        top_k=top_k
    )


print("✓ Retrieval Engine Ready")
print("=" * 70)

In [ ]:
# ================================================================
# CELL 13 — Generate Evaluation Dataset
# ================================================================

import random
from pathlib import Path
import pandas as pd

print("=" * 70)
print("Generating Evaluation Dataset")
print("=" * 70)

EVAL_FILE = Path(OUTPUT_DIR) / "evaluation_dataset.csv"
Path(OUTPUT_DIR).mkdir(exist_ok=True)

# ------------------------------------------------------------
# Query Generator
# ------------------------------------------------------------

def generate_queries(description):

    description = str(description).strip()

    return [
        description,
        f"Find BOQ item for {description}",
        f"Locate {description}",
        f"What is the BOQ entry for {description}?",
        f"Search item: {description}"
    ]


# ------------------------------------------------------------
# Query Difficulty
# ------------------------------------------------------------

def query_difficulty(query):

    words = len(query.split())

    if words <= 4:
        return "Easy"

    elif words <= 8:
        return "Medium"

    return "Hard"


# ------------------------------------------------------------
# Build Evaluation Dataset
# ------------------------------------------------------------

if EVAL_FILE.exists():

    print("Loading existing evaluation dataset...")

    eval_df = pd.read_csv(EVAL_FILE)

else:

    rows = []

    for idx, row in corpus_df.iterrows():

        desc = str(row["text"]).strip()

        if len(desc) < 5:
            continue

        queries = generate_queries(desc)

        for q in queries:

            rows.append({

                "query": q,

                "ground_truth": idx,

                "document": desc,

                "query_length": len(q),

                "word_count": len(q.split()),

                "query_difficulty": query_difficulty(q)

            })

    eval_df = pd.DataFrame(rows)

    eval_df.to_csv(EVAL_FILE, index=False)

print(f"Queries Generated : {len(eval_df):,}")
print(f"Ground Truth Docs : {eval_df['ground_truth'].nunique():,}")

print("\nQuery Difficulty Distribution")
print(eval_df["query_difficulty"].value_counts())

print("=" * 70)
print("✓ Evaluation Dataset Ready")
print("=" * 70)

In [ ]:
# =========================================================
# PART 1 — Metric Helpers
# =========================================================

import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

print("=" * 70)
print("Running PROFESSIONAL Benchmark Evaluation")
print("=" * 70)


# ---------------------------------------------------------
# Ranking Metrics
# ---------------------------------------------------------

def reciprocal_rank(ranks):
    if len(ranks) == 0:
        return 0.0
    return 1.0 / (ranks[0] + 1)


def average_precision(ranks):
    if len(ranks) == 0:
        return 0.0

    return np.mean([1.0 / (r + 1) for r in ranks])


# ---------------------------------------------------------
# Error Rate
# ---------------------------------------------------------

def error_rate(hit):

    return 0 if hit else 1


# ---------------------------------------------------------
# Query Difficulty
# ---------------------------------------------------------

def get_query_difficulty(query):

    words = len(str(query).split())

    if words <= 4:
        return "Easy"

    elif words <= 8:
        return "Medium"

    else:
        return "Hard"


# ---------------------------------------------------------
# Similarity Metrics
# ---------------------------------------------------------

def similarity_metrics(scores):

    return {

        "avg_similarity": float(np.mean(scores)),

        "max_similarity": float(np.max(scores)),

        "min_similarity": float(np.min(scores))

    }


# ---------------------------------------------------------
# Efficiency Metrics
# ---------------------------------------------------------

def efficiency_metrics(start_time, end_time):

    latency = (end_time - start_time) * 1000

    throughput = 1000 / latency if latency > 0 else 0

    return {

        "latency_ms": latency,

        "throughput_qps": throughput

    }


print("✓ Metric helper functions loaded.")

In [ ]:
# =========================================================
# PART 2 — FAST Evaluation Function (Batch Encoding)
# =========================================================

def evaluate(model, index):

    # ----------------------------------------
    # Encode ALL queries at once
    # ----------------------------------------

    queries = eval_df["query"].astype(str).tolist()
    ground_truth = eval_df["ground_truth"].astype(int).tolist()

    print(f"Encoding {len(queries):,} queries...")

    t0 = time.perf_counter()

    query_embeddings = model.encode(
        queries,
        batch_size=256,                 # Increase to 512 if GPU memory allows
        convert_to_numpy=True,
        normalize_embeddings=NORMALIZE,
        show_progress_bar=True
    ).astype(np.float32)

    encode_time = time.perf_counter() - t0

    print(f"Encoding completed in {encode_time:.2f} sec")

    # ----------------------------------------
    # Search ALL queries together
    # ----------------------------------------

    t0 = time.perf_counter()

    scores, ids = index.search(
        query_embeddings,
        max(TOP_K)
    )

    search_time = time.perf_counter() - t0

    print(f"FAISS Search completed in {search_time:.2f} sec")

    recalls = {k: [] for k in TOP_K}
    precisions = {k: [] for k in TOP_K}

    mrr_scores = []
    map_scores = []

    error_rates = []

    avg_similarity = []
    max_similarity = []
    min_similarity = []

    difficulty_stats = {
        "Easy": [],
        "Medium": [],
        "Hard": []
    }

    query_results = []

    latency_ms = search_time * 1000 / len(queries)
    throughput = len(queries) / search_time

    # ----------------------------------------
    # Metric Calculation Only
    # ----------------------------------------

    for i in tqdm(range(len(queries))):

        query = queries[i]
        gt = ground_truth[i]

        retrieved = ids[i]
        similarity = scores[i]

        difficulty = get_query_difficulty(query)

        ranks = np.where(retrieved == gt)[0]

        first_hit = False

        for k in TOP_K:

            topk = retrieved[:k]

            hit = gt in topk

            recalls[k].append(int(hit))
            precisions[k].append(int(hit) / k)

            if k == 1:
                first_hit = hit

        mrr_scores.append(reciprocal_rank(ranks))
        map_scores.append(average_precision(ranks))

        err = error_rate(first_hit)

        error_rates.append(err)
        difficulty_stats[difficulty].append(1 - err)

        sim = similarity_metrics(similarity)

        avg_similarity.append(sim["avg_similarity"])
        max_similarity.append(sim["max_similarity"])
        min_similarity.append(sim["min_similarity"])

        query_results.append({

            "query": query,
            "ground_truth": gt,
            "difficulty": difficulty,
            "hit": first_hit,
            "error_rate": err,

            "latency_ms": latency_ms,
            "throughput_qps": throughput,

            "avg_similarity": sim["avg_similarity"],
            "max_similarity": sim["max_similarity"],
            "min_similarity": sim["min_similarity"]

        })

    # ----------------------------------------
    # Final Results
    # ----------------------------------------

    results = {}

    for k in TOP_K:

        results[f"Recall@{k}"] = np.mean(recalls[k])
        results[f"Precision@{k}"] = np.mean(precisions[k])

    results["MRR"] = np.mean(mrr_scores)
    results["MAP"] = np.mean(map_scores)

    results["Error Rate"] = np.mean(error_rates)

    results["Latency(ms)"] = latency_ms
    results["Throughput(QPS)"] = throughput

    results["Average Similarity"] = np.mean(avg_similarity)
    results["Maximum Similarity"] = np.mean(max_similarity)
    results["Minimum Similarity"] = np.mean(min_similarity)

    results["Easy Accuracy"] = np.mean(difficulty_stats["Easy"]) if difficulty_stats["Easy"] else 0
    results["Medium Accuracy"] = np.mean(difficulty_stats["Medium"]) if difficulty_stats["Medium"] else 0
    results["Hard Accuracy"] = np.mean(difficulty_stats["Hard"]) if difficulty_stats["Hard"] else 0

    return results, pd.DataFrame(query_results)

print("✓ Ultra-Fast Evaluation Function Ready")

In [ ]:
# =========================================================
# PART 3 — Run Evaluation & Save Results
# =========================================================

import os
from pathlib import Path

# ---------------------------------------------------------
# Evaluate Base Model
# ---------------------------------------------------------

print("\nEvaluating BASE model...\n")

base_results, base_query_results = evaluate(
    base_model,
    base_index
)

# ---------------------------------------------------------
# Evaluate Fine-Tuned Model
# ---------------------------------------------------------

print("\nEvaluating Fine-Tuned model...\n")

ft_results, ft_query_results = evaluate(
    ft_model,
    ft_index
)

# ---------------------------------------------------------
# Comparison Table
# ---------------------------------------------------------

results_df = pd.DataFrame({

    "Metric": list(base_results.keys()),

    "Base Model": list(base_results.values()),

    "Fine-Tuned Model": list(ft_results.values())

})

results_df["Improvement"] = (

    results_df["Fine-Tuned Model"]

    -

    results_df["Base Model"]

)

results_df = results_df.round(4)

print("\nBenchmark Results\n")

display(results_df)

# ---------------------------------------------------------
# Query Difficulty Summary
# ---------------------------------------------------------

difficulty_summary = pd.DataFrame({

    "Difficulty": [

        "Easy",

        "Medium",

        "Hard"

    ],

    "Base Accuracy": [

        base_results["Easy Accuracy"],

        base_results["Medium Accuracy"],

        base_results["Hard Accuracy"]

    ],

    "Fine-Tuned Accuracy": [

        ft_results["Easy Accuracy"],

        ft_results["Medium Accuracy"],

        ft_results["Hard Accuracy"]

    ]

}).round(4)

print("\nQuery Difficulty Performance\n")

display(difficulty_summary)

# ---------------------------------------------------------
# Similarity Summary
# ---------------------------------------------------------

similarity_summary = pd.DataFrame({

    "Metric":[

        "Average Similarity",

        "Maximum Similarity",

        "Minimum Similarity"

    ],

    "Base":[

        base_results["Average Similarity"],

        base_results["Maximum Similarity"],

        base_results["Minimum Similarity"]

    ],

    "Fine-Tuned":[

        ft_results["Average Similarity"],

        ft_results["Maximum Similarity"],

        ft_results["Minimum Similarity"]

    ]

}).round(4)

print("\nSimilarity Metrics\n")

display(similarity_summary)

# ---------------------------------------------------------
# Efficiency Summary
# ---------------------------------------------------------

efficiency_summary = pd.DataFrame({

    "Metric":[

        "Latency (ms)",

        "Throughput (QPS)"

    ],

    "Base":[

        base_results["Latency(ms)"],

        base_results["Throughput(QPS)"]

    ],

    "Fine-Tuned":[

        ft_results["Latency(ms)"],

        ft_results["Throughput(QPS)"]

    ]

}).round(4)

print("\nEfficiency Metrics\n")

display(efficiency_summary)

# ---------------------------------------------------------
# Save Everything
# ---------------------------------------------------------

os.makedirs(OUTPUT_DIR, exist_ok=True)

table_dir = Path(OUTPUT_DIR) / "tables"

table_dir.mkdir(exist_ok=True)

results_df.to_csv(

    table_dir / "benchmark_results.csv",

    index=False

)

difficulty_summary.to_csv(

    table_dir / "query_difficulty_results.csv",

    index=False

)

similarity_summary.to_csv(

    table_dir / "similarity_metrics.csv",

    index=False

)

efficiency_summary.to_csv(

    table_dir / "efficiency_metrics.csv",

    index=False

)

base_query_results.to_csv(

    table_dir / "base_query_results.csv",

    index=False

)

ft_query_results.to_csv(

    table_dir / "ft_query_results.csv",

    index=False

)

# ---------------------------------------------------------
# Save Dictionaries
# ---------------------------------------------------------

np.save(

    Path(OUTPUT_DIR) / "base_results.npy",

    base_results

)

np.save(

    Path(OUTPUT_DIR) / "ft_results.npy",

    ft_results

)

print("\n" + "=" * 70)
print("✓ Professional Benchmark Evaluation Completed")
print("=" * 70)

print("\nSaved Files")

print("-" * 70)

print("benchmark_results.csv")

print("query_difficulty_results.csv")

print("similarity_metrics.csv")

print("efficiency_metrics.csv")

print("base_query_results.csv")

print("ft_query_results.csv")

print("base_results.npy")

print("ft_results.npy")

In [ ]:
# ================================================================
# CELL 14.5 — Validate Ground Truth
# ================================================================

print("=" * 60)
print("Ground Truth Validation")
print("=" * 60)

# ------------------------------------------------
# Build if Missing
# ------------------------------------------------

if "ground_truth" not in globals():
    ground_truth = list(range(len(corpus_df)))

# ------------------------------------------------
# Validation
# ------------------------------------------------

assert len(ground_truth) == len(corpus_df), \
    "Ground truth size does not match corpus."

assert len(queries) == len(ground_truth), \
    "Queries and ground truth have different lengths."

print(f"Corpus Documents : {len(corpus_df):,}")
print(f"Queries          : {len(queries):,}")
print(f"Ground Truth     : {len(ground_truth):,}")

print("=" * 60)
print("✓ Ground Truth Ready")
print("=" * 60)

In [ ]:
# ================================================================
# CELL 16 — Save Benchmark Results
# ================================================================

from pathlib import Path

print("=" * 70)
print("Saving Benchmark Results")
print("=" * 70)

# ------------------------------------------------
# Create Output Directories
# ------------------------------------------------

FIG_DIR = Path(OUTPUT_DIR) / "figures"
TABLE_DIR = Path(OUTPUT_DIR) / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------
# Save Main Benchmark Results
# ------------------------------------------------

results_df.to_csv(
    TABLE_DIR / "benchmark_results.csv",
    index=False
)

# ------------------------------------------------
# Save Query Difficulty Results
# ------------------------------------------------

if "difficulty_summary" in globals():

    difficulty_summary.to_csv(
        TABLE_DIR / "query_difficulty_results.csv",
        index=False
    )

# ------------------------------------------------
# Save Similarity Metrics
# ------------------------------------------------

if "similarity_summary" in globals():

    similarity_summary.to_csv(
        TABLE_DIR / "similarity_metrics.csv",
        index=False
    )

# ------------------------------------------------
# Save Efficiency Metrics
# ------------------------------------------------

if "efficiency_summary" in globals():

    efficiency_summary.to_csv(
        TABLE_DIR / "efficiency_metrics.csv",
        index=False
    )

# ------------------------------------------------
# Save Per-Query Results
# ------------------------------------------------

if "base_query_results" in globals():

    base_query_results.to_csv(
        TABLE_DIR / "base_query_results.csv",
        index=False
    )

if "ft_query_results" in globals():

    ft_query_results.to_csv(
        TABLE_DIR / "ft_query_results.csv",
        index=False
    )

# ------------------------------------------------
# Save Result Dictionaries
# ------------------------------------------------

if "base_results" in globals():
    np.save(TABLE_DIR / "base_results.npy", base_results)

if "ft_results" in globals():
    np.save(TABLE_DIR / "ft_results.npy", ft_results)

# ------------------------------------------------
# Summary
# ------------------------------------------------

print("✓ Benchmark Results Saved Successfully\n")

saved_files = [
    "benchmark_results.csv",
    "query_difficulty_results.csv",
    "similarity_metrics.csv",
    "efficiency_metrics.csv",
    "base_query_results.csv",
    "ft_query_results.csv",
    "base_results.npy",
    "ft_results.npy"
]

for file in saved_files:

    if (TABLE_DIR / file).exists():
        print(f"✓ {file}")

print("\nBenchmark Summary")
display(results_df)

if "difficulty_summary" in globals():
    print("\nQuery Difficulty Summary")
    display(difficulty_summary)

if "similarity_summary" in globals():
    print("\nSimilarity Metrics")
    display(similarity_summary)

if "efficiency_summary" in globals():
    print("\nEfficiency Metrics")
    display(efficiency_summary)

print("=" * 70)
print("✓ All benchmark outputs saved.")
print("=" * 70)

In [ ]:
# ================================================================
# CELL 17 — Publication Grade Retrieval Metric Comparison
# ================================================================

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

print("=" * 70)
print("Generating Retrieval Metric Comparison Plot")
print("=" * 70)

Path(FIG_DIR).mkdir(parents=True, exist_ok=True)

# ------------------------------------------------
# Keep only Retrieval Metrics
# ------------------------------------------------

retrieval_metrics = [

    "Recall@1",
    "Recall@3",
    "Recall@5",
    "Recall@10",

    "Precision@1",
    "Precision@3",
    "Precision@5",
    "Precision@10",

    "MRR",
    "MAP",
    "Error Rate"

]

plot_df = results_df[
    results_df["Metric"].isin(retrieval_metrics)
].copy()

metrics = plot_df["Metric"].tolist()

base_vals = plot_df["Base Model"].tolist()

ft_vals = plot_df["Fine-Tuned Model"].tolist()

# ------------------------------------------------
# Plot
# ------------------------------------------------

x = np.arange(len(metrics))
width = 0.35

plt.figure(figsize=(16,8))

bars1 = plt.bar(
    x - width/2,
    base_vals,
    width,
    label="Base Model"
)

bars2 = plt.bar(
    x + width/2,
    ft_vals,
    width,
    label="Fine-Tuned Model"
)

# ------------------------------------------------
# Labels
# ------------------------------------------------

for bars in [bars1, bars2]:

    for bar in bars:

        height = bar.get_height()

        plt.text(

            bar.get_x() + bar.get_width()/2,

            height,

            f"{height:.3f}",

            ha="center",

            va="bottom",

            fontsize=8

        )

# ------------------------------------------------
# Formatting
# ------------------------------------------------

plt.xticks(
    x,
    metrics,
    rotation=35,
    ha="right"
)

plt.ylabel("Score")

plt.title(
    "Retrieval Performance Comparison"
)

plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.4
)

plt.legend()

plt.tight_layout()

# ------------------------------------------------
# Save
# ------------------------------------------------

save_path = Path(FIG_DIR) / "retrieval_metrics_comparison.png"

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"✓ Saved : {save_path}")

In [ ]:
# ================================================================
# CELL 18 — Publication-Grade Recall@K Curve
# ================================================================

import matplotlib.pyplot as plt
from pathlib import Path

print("=" * 70)
print("Generating Recall@K Curve")
print("=" * 70)

# ------------------------------------------------
# Output Directory
# ------------------------------------------------

Path(FIG_DIR).mkdir(parents=True, exist_ok=True)

# ------------------------------------------------
# Extract Recall Metrics
# ------------------------------------------------

recall_df = results_df[
    results_df["Metric"].str.startswith("Recall@")
].copy()

recall_df["K"] = (
    recall_df["Metric"]
    .str.split("@")
    .str[1]
    .astype(int)
)

recall_df = recall_df.sort_values("K")

k_vals = recall_df["K"].tolist()

base_vals = recall_df["Base Model"].tolist()

ft_vals = recall_df["Fine-Tuned Model"].tolist()

# ------------------------------------------------
# Plot
# ------------------------------------------------

plt.figure(figsize=(9,6))

plt.plot(
    k_vals,
    base_vals,
    marker="o",
    linewidth=2.5,
    label="Base Model"
)

plt.plot(
    k_vals,
    ft_vals,
    marker="o",
    linewidth=2.5,
    label="Fine-Tuned Model"
)

# ------------------------------------------------
# Value Labels
# ------------------------------------------------

for x, y in zip(k_vals, base_vals):

    plt.text(
        x,
        y,
        f"{y:.3f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

for x, y in zip(k_vals, ft_vals):

    plt.text(
        x,
        y,
        f"{y:.3f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

# ------------------------------------------------
# Formatting
# ------------------------------------------------

plt.xticks(k_vals)

plt.xlabel("Top-K")

plt.ylabel("Recall")

plt.title("Recall@K Comparison")

plt.grid(
    linestyle="--",
    alpha=0.4
)

plt.legend()

plt.ylim(0, 1.05)

plt.tight_layout()

# ------------------------------------------------
# Save
# ------------------------------------------------

save_path = Path(FIG_DIR) / "recall_curve.png"

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"✓ Saved: {save_path}")

In [ ]:
# ================================================================
# CELL 19 — Publication-Grade Precision@K Curve
# ================================================================

import matplotlib.pyplot as plt
from pathlib import Path

print("=" * 70)
print("Generating Precision@K Curve")
print("=" * 70)

# ------------------------------------------------
# Output Directory
# ------------------------------------------------

Path(FIG_DIR).mkdir(parents=True, exist_ok=True)

# ------------------------------------------------
# Extract Precision Metrics
# ------------------------------------------------

precision_df = results_df[
    results_df["Metric"].str.startswith("Precision@")
].copy()

precision_df["K"] = (
    precision_df["Metric"]
    .str.split("@")
    .str[1]
    .astype(int)
)

precision_df = precision_df.sort_values("K")

k_vals = precision_df["K"].tolist()

base_vals = precision_df["Base Model"].tolist()

ft_vals = precision_df["Fine-Tuned Model"].tolist()

# ------------------------------------------------
# Plot
# ------------------------------------------------

plt.figure(figsize=(9,6))

plt.plot(
    k_vals,
    base_vals,
    marker="o",
    linewidth=2.5,
    label="Base Model"
)

plt.plot(
    k_vals,
    ft_vals,
    marker="o",
    linewidth=2.5,
    label="Fine-Tuned Model"
)

# ------------------------------------------------
# Value Labels
# ------------------------------------------------

for x, y in zip(k_vals, base_vals):

    plt.text(
        x,
        y,
        f"{y:.3f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

for x, y in zip(k_vals, ft_vals):

    plt.text(
        x,
        y,
        f"{y:.3f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

# ------------------------------------------------
# Formatting
# ------------------------------------------------

plt.xticks(k_vals)

plt.xlabel("Top-K")

plt.ylabel("Precision")

plt.title("Precision@K Comparison")

plt.grid(
    linestyle="--",
    alpha=0.4
)

plt.legend()

plt.tight_layout()

# ------------------------------------------------
# Save
# ------------------------------------------------

save_path = Path(FIG_DIR) / "precision_curve.png"

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"✓ Saved: {save_path}")

In [ ]:
# ================================================================
# CELL 20 — Publication Grade Retrieval Benchmark Comparison
# ================================================================

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("=" * 70)
print("Generating Publication-Grade Retrieval Benchmark Comparison")
print("=" * 70)

Path(FIG_DIR).mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------
# Retrieval Metrics Only
# --------------------------------------------------------

retrieval_metrics = [

    "Recall@1",
    "Recall@3",
    "Recall@5",
    "Recall@10",

    "Precision@1",
    "Precision@3",
    "Precision@5",
    "Precision@10",

    "MRR",
    "MAP",

    "Error Rate"

]

plot_df = results_df[
    results_df["Metric"].isin(retrieval_metrics)
].copy()

# --------------------------------------------------------
# Improvement %
# --------------------------------------------------------

plot_df["Improvement (%)"] = (

    (plot_df["Fine-Tuned Model"] -

     plot_df["Base Model"])

    /

    plot_df["Base Model"].replace(0, np.nan)

) * 100

plot_df["Improvement (%)"] = (
    plot_df["Improvement (%)"]
    .fillna(0)
    .round(2)
)

metrics = plot_df["Metric"].tolist()

base_vals = plot_df["Base Model"].tolist()

ft_vals = plot_df["Fine-Tuned Model"].tolist()

improve = plot_df["Improvement (%)"].tolist()

# --------------------------------------------------------
# Plot
# --------------------------------------------------------

x = np.arange(len(metrics))

width = 0.38

plt.figure(figsize=(18,8))

bars1 = plt.bar(

    x - width/2,

    base_vals,

    width,

    label="Base Model",

    edgecolor="black"

)

bars2 = plt.bar(

    x + width/2,

    ft_vals,

    width,

    label="Fine-Tuned Model",

    edgecolor="black"

)

# --------------------------------------------------------
# Value Labels
# --------------------------------------------------------

for bars in [bars1, bars2]:

    for bar in bars:

        height = bar.get_height()

        plt.text(

            bar.get_x() + bar.get_width()/2,

            height,

            f"{height:.3f}",

            ha="center",

            va="bottom",

            fontsize=8

        )

# --------------------------------------------------------
# Improvement Labels
# --------------------------------------------------------

for i in range(len(metrics)):

    ymax = max(

        base_vals[i],

        ft_vals[i]

    )

    color = "green"

    if improve[i] < 0:
        color = "red"

    plt.text(

        x[i],

        ymax + 0.03,

        f"{improve[i]:+.1f}%",

        ha="center",

        fontsize=10,

        fontweight="bold",

        color=color

    )

# --------------------------------------------------------
# Formatting
# --------------------------------------------------------

plt.xticks(

    x,

    metrics,

    rotation=35,

    ha="right"

)

plt.ylabel("Metric Score")

plt.title(

    "Retrieval Benchmark Comparison\n(Base Model vs Fine-Tuned Model)",

    fontsize=15,

    weight="bold"

)

plt.grid(

    axis="y",

    linestyle="--",

    alpha=0.35

)

plt.legend()

plt.tight_layout()

# --------------------------------------------------------
# Save
# --------------------------------------------------------

save_path = Path(FIG_DIR) / "benchmark_comparison_professional.png"

plt.savefig(

    save_path,

    dpi=400,

    bbox_inches="tight"

)

plt.show()

print(f"✓ Saved: {save_path}")

In [ ]:
# ================================================================
# CELL 21 — Publication-Grade Retrieval Radar Chart
# ================================================================

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("=" * 70)
print("Generating Retrieval Radar Chart")
print("=" * 70)

Path(FIG_DIR).mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------
# Retrieval Metrics Only
# --------------------------------------------------------

retrieval_metrics = [

    "Recall@1",
    "Recall@3",
    "Recall@5",
    "Recall@10",

    "Precision@1",
    "Precision@3",
    "Precision@5",
    "Precision@10",

    "MRR",
    "MAP",

    "Error Rate"

]

radar = results_df[
    results_df["Metric"].isin(retrieval_metrics)
].copy()

labels = radar["Metric"].tolist()

base = radar["Base Model"].to_numpy(dtype=float)

ft = radar["Fine-Tuned Model"].to_numpy(dtype=float)

# --------------------------------------------------------
# Normalize Values
# --------------------------------------------------------

max_vals = np.maximum(base, ft)

max_vals[max_vals == 0] = 1

base_norm = base / max_vals

ft_norm = ft / max_vals

# --------------------------------------------------------
# Close Radar
# --------------------------------------------------------

angles = np.linspace(
    0,
    2 * np.pi,
    len(labels),
    endpoint=False
)

angles = np.concatenate(
    [angles, [angles[0]]]
)

base_norm = np.concatenate(
    [base_norm, [base_norm[0]]]
)

ft_norm = np.concatenate(
    [ft_norm, [ft_norm[0]]]
)

# --------------------------------------------------------
# Plot
# --------------------------------------------------------

fig = plt.figure(figsize=(9,9))

ax = plt.subplot(
    111,
    polar=True
)

ax.plot(
    angles,
    base_norm,
    linewidth=2,
    label="Base Model"
)

ax.fill(
    angles,
    base_norm,
    alpha=0.25
)

ax.plot(
    angles,
    ft_norm,
    linewidth=2,
    label="Fine-Tuned Model"
)

ax.fill(
    angles,
    ft_norm,
    alpha=0.25
)

# --------------------------------------------------------
# Formatting
# --------------------------------------------------------

ax.set_xticks(
    angles[:-1]
)

ax.set_xticklabels(
    labels,
    fontsize=10
)

ax.set_ylim(0,1)

ax.set_yticks(
    [0.2,0.4,0.6,0.8,1.0]
)

ax.set_yticklabels(
    ["0.2","0.4","0.6","0.8","1.0"]
)

plt.title(
    "Normalized Retrieval Performance Radar Chart",
    fontsize=15,
    weight="bold"
)

plt.legend(
    loc="upper right",
    bbox_to_anchor=(1.25,1.15)
)

plt.tight_layout()

# --------------------------------------------------------
# Save
# --------------------------------------------------------

save_path = Path(FIG_DIR) / "retrieval_radar_chart.png"

plt.savefig(
    save_path,
    dpi=400,
    bbox_inches="tight"
)

plt.show()

print(f"✓ Saved: {save_path}")

In [ ]:
# ================================================================
# CELL 22 — Comprehensive Benchmark Tables
# ================================================================

import pandas as pd
from pathlib import Path

print("=" * 80)
print("Generating Comprehensive Benchmark Tables")
print("=" * 80)

TABLE_DIR = Path(OUTPUT_DIR) / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# ================================================================
# TABLE 1 — Benchmark Metric Summary
# ================================================================

metric_description = {

    "Recall@1":
        "Correct document retrieved at Rank 1.",

    "Recall@3":
        "Correct document appears within Top-3 results.",

    "Recall@5":
        "Correct document appears within Top-5 results.",

    "Recall@10":
        "Correct document appears within Top-10 results.",

    "Precision@1":
        "Proportion of relevant documents in Top-1.",

    "Precision@3":
        "Proportion of relevant documents in Top-3.",

    "Precision@5":
        "Proportion of relevant documents in Top-5.",

    "Precision@10":
        "Proportion of relevant documents in Top-10.",

    "MRR":
        "Mean Reciprocal Rank of the first correct retrieval.",

    "MAP":
        "Mean Average Precision across all queries.",

    "Error Rate":
        "Fraction of queries where the correct document was not retrieved at Rank 1.",

    "Average Similarity":
        "Average cosine similarity of retrieved documents.",

    "Maximum Similarity":
        "Highest similarity score among retrieved documents.",

    "Minimum Similarity":
        "Lowest similarity score among retrieved documents.",

    "Latency(ms)":
        "Average time required to answer one query.",

    "Throughput(QPS)":
        "Queries processed per second.",

    "Easy Accuracy":
        "Retrieval accuracy for easy queries.",

    "Medium Accuracy":
        "Retrieval accuracy for medium-complexity queries.",

    "Hard Accuracy":
        "Retrieval accuracy for difficult queries."

}

lower_is_better = {

    "Error Rate",

    "Latency(ms)"

}

summary_rows = []

for _, row in results_df.iterrows():

    metric = row["Metric"]

    summary_rows.append({

        "Metric": metric,

        "Description":
            metric_description.get(metric, "-"),

        "Base Model":
            round(row["Base Model"], 4),

        "Fine-Tuned Model":
            round(row["Fine-Tuned Model"], 4),

        "Improvement":
            round(row["Improvement"], 4),

        "Interpretation":
            "Lower is Better"
            if metric in lower_is_better
            else "Higher is Better"

    })

metric_summary_df = pd.DataFrame(summary_rows)

print("\nTABLE 1 : Benchmark Metric Summary")

display(metric_summary_df)

metric_summary_df.to_csv(

    TABLE_DIR /
    "Table_1_Benchmark_Metrics.csv",

    index=False

)

# ================================================================
# TABLE 2 — Query Difficulty Performance
# ================================================================

if "difficulty_summary" in globals():

    print("\nTABLE 2 : Query Difficulty")

    display(difficulty_summary)

    difficulty_summary.to_csv(

        TABLE_DIR /
        "Table_2_Query_Difficulty.csv",

        index=False

    )

# ================================================================
# TABLE 3 — Similarity Metrics
# ================================================================

if "similarity_summary" in globals():

    print("\nTABLE 3 : Similarity Metrics")

    display(similarity_summary)

    similarity_summary.to_csv(

        TABLE_DIR /
        "Table_3_Similarity_Metrics.csv",

        index=False

    )

# ================================================================
# TABLE 4 — Efficiency Metrics
# ================================================================

if "efficiency_summary" in globals():

    print("\nTABLE 4 : Efficiency Metrics")

    display(efficiency_summary)

    efficiency_summary.to_csv(

        TABLE_DIR /
        "Table_4_Efficiency_Metrics.csv",

        index=False

    )

# ================================================================
# TABLE 5 — Popular Embedding Models
# ================================================================

models_df = pd.DataFrame([

["Qwen3-Embedding-0.6B (Base)","Alibaba","1024","Multilingual","Baseline embedding model"],

["Qwen3-Embedding-0.6B (Fine-Tuned)","Alibaba","1024","Multilingual","Fine-tuned on BOQ corpus"],

["BGE-M3","BAAI","1024","Multilingual","General retrieval model"],

["BGE-Large-EN-v1.5","BAAI","1024","English","Large English embedding model"],

["e5-large-v2","Microsoft","1024","Multilingual","Instruction-tuned embedding"],

["multilingual-e5-large","Microsoft","1024","Multilingual","Cross-lingual retrieval"],

["all-mpnet-base-v2","Sentence Transformers","768","English","Semantic sentence embeddings"],

["all-MiniLM-L6-v2","Sentence Transformers","384","English","Fast lightweight model"],

["gte-large-en-v1.5","Alibaba","1024","English","Dense retrieval model"],

["jina-embeddings-v3","Jina AI","1024","Multilingual","Long-context retrieval"],

["NV-Embed-v2","NVIDIA","4096","English","Large retrieval embedding"]

],

columns=[

"Embedding Model",

"Developer",

"Embedding Dimension",

"Language Support",

"Description"

])

print("\nTABLE 5 : Popular Embedding Models")

display(models_df)

models_df.to_csv(

    TABLE_DIR /
    "Table_5_Embedding_Models.csv",

    index=False

)

print("\n" + "=" * 80)
print("✓ All benchmark tables generated successfully.")
print("=" * 80)

In [ ]:
# ================================================================
# CELL 23 — GROQ LLM GROUNDED BENCHMARK SUMMARY (FIXED)
# ================================================================

try:
    from groq import Groq
except ImportError:
    !pip -q install groq
    from groq import Groq

import os
from pathlib import Path

print("=" * 80)
print("Generating GROQ LLM Evaluation Summary (Grounded)")
print("=" * 80)

# -------------------------------------------------------
# SECURITY FIX (DO NOT HARDCODE IN FINAL PROJECT)
# -------------------------------------------------------


client = Groq(api_key="YOUR_GROQ_API_KEY")

if GROQ_API_KEY is None:
    raise ValueError(
        "❌ GROQ_API_KEY not found. Please set it using environment variables."
    )

client = Groq(api_key=GROQ_API_KEY)

# -------------------------------------------------------
# SAFE METRICS INPUT (structured instead of raw markdown)
# -------------------------------------------------------

metrics_text = results_df.to_string(index=False)

# -------------------------------------------------------
# GROUNDED PROMPT
# -------------------------------------------------------

# -------------------------------------------------------
# GROUNDED PROMPT
# -------------------------------------------------------

prompt = f"""
You are an expert research scientist writing a dissertation-level evaluation report.

You are given STRICT benchmark results comparing two embedding models:

• Base Model
• Fine-Tuned Model

Use ONLY the numerical values provided.
Do NOT invent any numbers.
Do NOT estimate missing values.
Do NOT assume improvements that are not shown.

==================================================
BENCHMARK RESULTS
==================================================

{metrics_text}

==================================================
WRITE THE REPORT USING ONLY THESE RESULTS
==================================================

Generate the following sections.

# Executive Summary

Write 5–6 concise bullet points summarizing the overall benchmark findings using exact values whenever possible.

# Retrieval Performance

Analyse:

- Recall@K
- Precision@K
- MRR
- MAP
- Error Rate

Discuss ranking quality and retrieval effectiveness using only the provided metrics.

# Similarity Analysis

Discuss:

- Average Similarity
- Maximum Similarity
- Minimum Similarity

Explain what these values indicate regarding semantic retrieval quality.

# Efficiency Analysis

Discuss:

- Latency
- Throughput

Compare computational efficiency between the two models.

# Query Difficulty Analysis

Analyse performance on:

- Easy Queries
- Medium Queries
- Hard Queries

Identify where each model performs best or worst.

# Key Improvements

Write bullet points only.

Include only measurable improvements supported by the benchmark.

# Limitations

Mention:

- metrics with very small improvements
- degraded metrics
- remaining weaknesses

# Final Conclusion

Write one dissertation-quality academic conclusion based strictly on the benchmark results.

Style:

- Formal
- Scientific
- Neutral
- Evidence-based
- No marketing language
- No assumptions
"""

# -------------------------------------------------------
# API CALL
# -------------------------------------------------------

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "system",
            "content": "You are a strict scientific evaluator. You only use provided numerical evidence."
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.1,
    max_tokens=1500
)

summary = response.choices[0].message.content

# -------------------------------------------------------
# DISPLAY
# -------------------------------------------------------

print(summary)

# -------------------------------------------------------
# SAVE OUTPUT
# -------------------------------------------------------

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

summary_file = output_dir / "evaluation_summary.txt"

with open(summary_file, "w", encoding="utf-8") as f:
    f.write(summary)

print("\n✓ Saved:", summary_file)

In [ ]:
# ================================================================
# FINAL MASTER CELL — COMPLETE BENCHMARK EXPORT
# ================================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path

print("="*80)
print("EXPORTING COMPLETE BENCHMARK RESULTS")
print("="*80)

# ----------------------------------------------------------
# Directories
# ----------------------------------------------------------

OUTPUT_PATH = Path(OUTPUT_DIR)
TABLE_DIR = OUTPUT_PATH / "tables"

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# Copy Results
# ----------------------------------------------------------

eval_table = results_df.copy()

# ----------------------------------------------------------
# Relative Improvement (%)
# ----------------------------------------------------------

eval_table["Improvement (%)"] = (

    (
        eval_table["Fine-Tuned Model"] -
        eval_table["Base Model"]
    )

    /

    eval_table["Base Model"].replace(0, np.nan)

) * 100

eval_table["Improvement (%)"] = (

    eval_table["Improvement (%)"]

    .replace([np.inf, -np.inf], 0)

    .fillna(0)

    .round(3)

)

print("\nBenchmark Results\n")

display(eval_table)

# ----------------------------------------------------------
# Overall Statistics
# ----------------------------------------------------------

best_row = eval_table.loc[
    eval_table["Improvement"].idxmax()
]

worst_row = eval_table.loc[
    eval_table["Improvement"].idxmin()
]

summary = {

    "Total Metrics":

        int(len(eval_table)),

    "Best Improving Metric":

        best_row["Metric"],

    "Worst Improving Metric":

        worst_row["Metric"],

    "Average Absolute Improvement":

        float(eval_table["Improvement"].mean()),

    "Average Relative Improvement (%)":

        float(eval_table["Improvement (%)"].mean())

}

print("\nSummary Statistics\n")

print(json.dumps(summary, indent=4))

# ----------------------------------------------------------
# Metric Dictionary
# ----------------------------------------------------------

metric_dictionary = {}

for _, row in eval_table.iterrows():

    metric_dictionary[row["Metric"]] = {

        "Base Model":

            float(row["Base Model"]),

        "Fine-Tuned Model":

            float(row["Fine-Tuned Model"]),

        "Absolute Improvement":

            float(row["Improvement"]),

        "Relative Improvement (%)":

            float(row["Improvement (%)"])

    }

# ----------------------------------------------------------
# Save Benchmark Table
# ----------------------------------------------------------

eval_table.to_csv(

    TABLE_DIR /

    "FULL_EVALUATION_RESULTS.csv",

    index=False

)

# ----------------------------------------------------------
# Save Metric Dictionary
# ----------------------------------------------------------

with open(

    TABLE_DIR /

    "FULL_EVALUATION_RESULTS.json",

    "w"

) as f:

    json.dump(

        metric_dictionary,

        f,

        indent=4

    )

# ----------------------------------------------------------
# Save Summary
# ----------------------------------------------------------

with open(

    TABLE_DIR /

    "EVALUATION_SUMMARY.json",

    "w"

) as f:

    json.dump(

        summary,

        f,

        indent=4

    )

# ----------------------------------------------------------
# Export Optional Tables
# ----------------------------------------------------------

optional_tables = {

    "difficulty_summary":
        "QUERY_DIFFICULTY_RESULTS.csv",

    "similarity_summary":
        "SIMILARITY_RESULTS.csv",

    "efficiency_summary":
        "EFFICIENCY_RESULTS.csv"

}

for var_name, filename in optional_tables.items():

    if var_name in globals():

        globals()[var_name].to_csv(

            TABLE_DIR / filename,

            index=False

        )

# ----------------------------------------------------------
# Export Individual Metric Categories
# ----------------------------------------------------------

retrieval_metrics = eval_table[
    eval_table["Metric"].str.contains(
        "Recall|Precision|MRR|MAP|Error",
        case=False,
        regex=True
    )
]

retrieval_metrics.to_csv(

    TABLE_DIR /

    "RETRIEVAL_METRICS.csv",

    index=False

)

similarity_metrics = eval_table[
    eval_table["Metric"].str.contains(
        "Similarity",
        case=False
    )
]

if len(similarity_metrics):

    similarity_metrics.to_csv(

        TABLE_DIR /

        "SIMILARITY_METRICS.csv",

        index=False

    )

efficiency_metrics = eval_table[
    eval_table["Metric"].str.contains(
        "Latency|Throughput",
        case=False
    )
]

if len(efficiency_metrics):

    efficiency_metrics.to_csv(

        TABLE_DIR /

        "EFFICIENCY_METRICS.csv",

        index=False

    )

difficulty_metrics = eval_table[
    eval_table["Metric"].str.contains(
        "Easy|Medium|Hard",
        case=False
    )
]

if len(difficulty_metrics):

    difficulty_metrics.to_csv(

        TABLE_DIR /

        "QUERY_DIFFICULTY_METRICS.csv",

        index=False

    )

# ----------------------------------------------------------
# Finish
# ----------------------------------------------------------

print("\n" + "="*80)
print("✓ COMPLETE BENCHMARK EXPORT FINISHED")
print("="*80)

print("\nFiles Generated")

for file in sorted(TABLE_DIR.glob("*")):

    print("•", file.name)

print("\nOverall Summary")

print(f"Total Metrics               : {summary['Total Metrics']}")
print(f"Best Improvement            : {summary['Best Improving Metric']}")
print(f"Worst Improvement           : {summary['Worst Improving Metric']}")
print(f"Average Improvement         : {summary['Average Absolute Improvement']:.4f}")
print(f"Average Relative Gain (%)   : {summary['Average Relative Improvement (%)']:.2f}%")

print("="*80)

In [ ]:
# Report Generation.

In [ ]:
# ================================================================
# FINAL CELL — PURE LLM-DRIVEN RESEARCH REPORT GENERATOR
# ================================================================

import os
from pathlib import Path
from groq import Groq
from docx import Document
from docx.shared import Inches, Pt

print("=" * 80)
print("LLM-DRIVEN REPORT GENERATOR (NO MANUAL TEXT)")
print("=" * 80)

# -----------------------------------------------------------------
# API KEY
# Replace with your NEW Groq API key
# -----------------------------------------------------------------

os.environ["YOUR_GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

GROQ_API_KEY = os.getenv("YOUR_GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("Groq API key not found.")

client = Groq(api_key=GROQ_API_KEY)

# -----------------------------------------------------------------
# CHECK REQUIRED VARIABLES
# -----------------------------------------------------------------

required = ["results_df", "FIG_DIR", "OUTPUT_DIR"]

for var in required:
    if var not in globals():
        raise NameError(f"{var} is not defined.")

# -----------------------------------------------------------------
# INPUT DATA
# -----------------------------------------------------------------

metrics_text = results_df.to_string(index=False)

figures = sorted(Path(FIG_DIR).glob("*.png"))

figure_list = "\n".join(
    f"- {fig.name}" for fig in figures
)

# -----------------------------------------------------------------
# PROMPT
# -----------------------------------------------------------------

prompt = f"""
You are writing a complete academic research paper based ONLY on the supplied experimental results.

You must NOT use outside knowledge.
You must NOT invent values.
You must ONLY discuss the supplied metrics.

=========================
METRICS
=========================

{metrics_text}

=========================
AVAILABLE FIGURES
=========================

{figure_list}

=========================
WRITE

1. Title

2. Abstract

3. Methodology

4. Experimental Setup

5. Results Analysis
   - Mention actual metric values.
   - Compare models using the provided numbers.

6. Discussion

7. Conclusion

Rules:
- Academic writing style.
- No hallucinated information.
- Reference figure filenames where appropriate.
- 6–12 sentences per section.
"""

# -----------------------------------------------------------------
# CALL GROQ
# -----------------------------------------------------------------

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "system",
            "content": (
                "You are a strict academic paper generator. "
                "Only use the supplied experimental data."
            ),
        },
        {
            "role": "user",
            "content": prompt,
        },
    ],
    temperature=0.1,
    max_tokens=2500,
)

paper_text = response.choices[0].message.content

# -----------------------------------------------------------------
# CREATE WORD DOCUMENT
# -----------------------------------------------------------------

doc = Document()

title = doc.add_heading("LLM Generated Research Paper", level=0)
title.alignment = 1

for line in paper_text.split("\n"):

    line = line.strip()

    if not line:
        continue

    # Detect headings
    if (
        line.endswith(":")
        or line.lower().startswith("title")
        or line.lower().startswith("abstract")
        or line.lower().startswith("methodology")
        or line.lower().startswith("experimental")
        or line.lower().startswith("results")
        or line.lower().startswith("discussion")
        or line.lower().startswith("conclusion")
    ):
        doc.add_heading(line.replace(":", ""), level=1)

    else:
        p = doc.add_paragraph()
        run = p.add_run(line)
        run.font.size = Pt(11)

# -----------------------------------------------------------------
# ADD FIGURES
# -----------------------------------------------------------------

if figures:

    doc.add_page_break()
    doc.add_heading("Figures", level=1)

    for fig in figures:

        doc.add_paragraph(fig.name)

        try:
            doc.add_picture(str(fig), width=Inches(6))

        except Exception as e:
            print(f"Could not insert {fig.name}: {e}")

# -----------------------------------------------------------------
# SAVE
# -----------------------------------------------------------------

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

output_file = Path(OUTPUT_DIR) / "LLM_Research_Paper.docx"

doc.save(output_file)

print("=" * 80)
print("✓ Research paper generated successfully.")
print("Saved to:", output_file)
print("=" * 80)

In [ ]:
out_path = Path(OUTPUT_DIR) / "LLM_Research_Paper.docx"
doc.save(out_path)

In [ ]:
from google.colab import files

files.download(str(file_path))